# 🩺 Silent Risk — Predicting Chronic Disease Before It Strikes

---

> **Context:** A national health insurer wants to identify high-risk members before they develop costly chronic conditions. This project leverages the CDC BRFSS 2024 survey (~400k respondents) to predict diabetes risk and multi-morbidity, and to map prevention priorities geographically.

---

## Section 0 — Introduction & Business Context

---

### Section 0.1 — Project Pitch

#### The Problem

Health insurers operate in a fundamentally reactive model: claims arrive, costs are absorbed, and interventions come too late. Chronic diseases — diabetes, heart disease, depression, COPD — develop silently over years before they surface as expensive medical events.

#### The Opportunity

The CDC BRFSS survey captures exactly the behavioral and demographic signals that precede chronic disease: smoking habits, physical inactivity, income stress, lack of healthcare access. These signals are observable **before** diagnosis.

#### The Solution

A predictive risk engine that:
- Scores individuals by probability of diabetes and multi-morbidity
- Segments the population into actionable risk tiers
- Maps prevention priorities geographically by state
- Identifies the highest-risk undetected profiles for proactive outreach

#### Business Value

| Stakeholder | Value |
|---|---|
| Actuaries | Earlier risk pricing and reserving |
| Prevention teams | Targeted outreach to high-risk members |
| Medical direction | Evidence-based program design |
| Regional agencies | Geographic prioritization of resources |

### Section 0.2 — Dataset Architecture

The BRFSS is a state-based surveillance system collecting data on health risk behaviors, chronic conditions, and use of preventive services among US adults. Data is collected via telephone interviews across all 50 states and territories.

The 2024 dataset is organized into thematic sections:

| Section | Content | Variables |
|---|---|---|
| Record Identification | State, date, sampling unit | 9 |
| Health Status | General health, healthy days | 4 |
| Healthcare Access | Insurance, provider, cost barriers | 4 |
| Chronic Conditions | Diabetes, heart disease, cancer, COPD, depression, kidney disease, arthritis | 13 |
| Demographics | Age, sex, income, education, employment, marital status | 13 |
| Behavioral Risk | Tobacco, alcohol, physical activity | 9 |
| Disability | Hearing, vision, mobility, cognition | 6 |
| Cancer Screening | Breast, cervical, colorectal, lung | 28 |
| Immunization | Flu shot, pneumonia vaccine | 4 |
| HIV/AIDS | Testing, risk factors | 3 |

**Key design note.** BRFSS uses special codes for missing data that must be handled carefully:

| Code | Meaning |
|---|---|
| `7` / `77` / `777` | Don't know / Not sure |
| `9` / `99` / `999` | Refused |
| `BLANK` | Not asked or missing |

### Section 0.3 — Setup: Imports & Data Loading

In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, RocCurveDisplay,
                             PrecisionRecallDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import lightgbm as lgb
import shap
import sqlite3

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)

# Load dataset
DATA_PATH = '/Users/arvind.b/brfss-data/LLCP2024.csv'
df = pd.read_csv(DATA_PATH)

print(f"✅ Dataset loaded successfully")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]:,}")

/Users/arvind.b/anaconda3/envs/silent-risk/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dataset loaded successfully
   Rows    : 457,670
   Columns : 301
